# Notebook 9 - Categorical Data Analysis
Sprint 4 - EDA

In this notebook I study the categorical features of my dataset in detail: the unique categories, their frequencies and distribution, rare categories, cardinality and category imbalance. Then I study how the important categorical features relate to the target Churn, and how they relate to each other. The aim is to decide later how each categorical feature should be encoded and which ones are likely to be useful.

**Dataset:** Telco Customer Churn (Kaggle, 7043 rows and 21 columns)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in df.columns if c not in num_cols + ["customerID", "Churn"]]

# working copy for analysis only
df_cat = df[cat_cols].copy()
df_cat["SeniorCitizen"] = df_cat["SeniorCitizen"].map({0: "No", 1: "Yes"})   # readable labels
df_cat["Churn"] = df["Churn"]
df_cat["Churn_num"] = (df["Churn"] == "Yes").astype(int)

print("Categorical features:", len(cat_cols))
print(cat_cols)

SeniorCitizen is stored as 0 and 1 but it is a category, so in this working copy I changed its labels to No and Yes to make the tables easier to read. The original data is not changed.

## 1. Unique Categories

**Concept:** The unique categories of a feature are the different labels it contains. The number of unique categories is called its cardinality. Listing them shows what the feature can say, and helps to find wrong or inconsistent labels.

**Example (business):** A payment method column in a shop has the unique categories UPI, Card and Cash. If the list also contained Upi and upi, they would be spelling variants of the same category and would need cleaning.

In [ ]:
for col in cat_cols:
    print(col, "(", df_cat[col].nunique(), "categories ):", sorted(df_cat[col].unique().tolist()))

**What the code does:** For every categorical feature it prints the number of unique categories and the list of the labels in alphabetical order.

**Why this method was used:** Reading the actual labels is the most direct way to understand what each feature contains and to spot spelling or format problems.

**What the output means:** Most features have 2 categories (Yes and No, or Male and Female). MultipleLines, InternetService, Contract, and the six add-on services have 3, and PaymentMethod has 4. Several features have the labels No internet service or No phone service as a third category.

**Insights:** The labels are clean, with no spelling variants, which confirms the data quality notebook. The only point to think about is that the third label No internet service and No phone service appears in seven features, and it needs a decision during encoding.

**Effect on the ML pipeline:** The list gives the number of columns that one-hot encoding will create, and which features can be encoded as simple 0 and 1.

## 2. Category Frequencies and Distribution

**Concept:** The frequency of a category is how many rows have that label, and the distribution is the share of each category. A distribution can be even (all categories have similar shares) or dominated by one category. A simple measure of evenness is the ratio between the largest and the smallest category count, and another is the normalized entropy, which is 1 when all categories have the same share and goes toward 0 when one category dominates.

**Example (AI/ML):** In a feature with 99% of one label and 1% of another, the feature is almost constant, so it can hardly help a model to separate the classes, even if the small group is very different.

In [ ]:
rows = []
for col in cat_cols:
    counts = df_cat[col].value_counts()
    share = counts / counts.sum()
    entropy = -(share * np.log(share)).sum() / np.log(len(counts))     # 1 means perfectly even
    rows.append([col, len(counts), counts.index[0], round(share.iloc[0] * 100, 1),
                 counts.index[-1], round(share.iloc[-1] * 100, 1),
                 round(counts.iloc[0] / counts.iloc[-1], 2), round(entropy, 2)])

dist_table = pd.DataFrame(rows, columns=["feature", "categories", "largest_category", "largest_%",
                                         "smallest_category", "smallest_%", "largest_to_smallest", "evenness"])
dist_table.sort_values("evenness")

In [ ]:
# frequency table of every category (used later)
freq_rows = []
for col in cat_cols:
    counts = df_cat[col].value_counts()
    for category, count in counts.items():
        freq_rows.append([col, category, count, round(count / len(df_cat) * 100, 2)])

freq_all = pd.DataFrame(freq_rows, columns=["feature", "category", "count", "percent"])
freq_all.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_data = dist_table.sort_values("largest_%")
ax.barh(plot_data["feature"], plot_data["largest_%"], color="cornflowerblue")
ax.axvline(80, color="red", linestyle="--", label="80 percent")
ax.set_xlabel("Share of the largest category (%)")
ax.set_title("How dominant is the largest category?")
ax.legend()
plt.tight_layout()
plt.show()

**What the code does:** The first cell builds one row per feature with the number of categories, the largest and smallest category with their shares, the ratio between them and the normalized entropy. The second cell builds a long table of the frequency of every category in every feature. The third cell plots the share of the largest category of each feature with a line at 80%.

**Why this method was used:** One summary table puts all sixteen features on the same scale, and the plot shows quickly which features are dominated by one label, so I do not need to read sixteen separate value counts.

**What the output means:** Gender has a share of about 50% for each label and an evenness close to 1, so it is almost perfectly balanced. PhoneService is the most dominated feature, with about 90% of customers having phone service. SeniorCitizen is also dominated, with about 84% non senior. Most of the other features have a largest category between about 40% and 70%.

**Insights:** Most features are reasonably balanced, and only PhoneService and SeniorCitizen cross the 80% line. A feature dominated by one label has less information, and PhoneService is a good candidate for being weak, which I check in the relationship with the target below.

**Effect on the ML pipeline:** Strongly dominated features give the model little to learn from, unless the small group behaves very differently from the large group. They may be removed in feature selection if they show no relationship with the target.

## 3. Rare Categories

**Concept:** A rare category is a label that appears in only a small share of the rows, for example under 1% or 5%. Rare categories are a problem because a model has very few examples to learn from, a train and test split may put them only in one part, and they create extra columns in one-hot encoding. Common treatments are to group them into an Other category, or to merge them with a similar category.

**Example (business):** In a column with 300 cities, 200 of them may each appear in fewer than 10 customer records. Their churn rates are very unreliable, because one churned customer changes the rate a lot, so it is better to group them as Other.

In [ ]:
for threshold in [1, 5, 10]:
    rare = freq_all[freq_all["percent"] < threshold]
    print("Categories with a share below", threshold, "%:", len(rare))

print()
print("Smallest categories in the dataset:")
freq_all.sort_values("percent").head(8)

**What the code does:** It counts how many categories have a share below 1%, 5% and 10% of all rows, and shows the eight smallest categories across all features.

**Why this method was used:** Using three thresholds shows how sensitive the result is to the definition of rare. The list of the smallest categories shows exactly which ones are close to being rare.

**What the output means:** There are no categories below 1% or 5%. Two categories fall below 10%: customers without phone service (PhoneService = No, and the matching label in MultipleLines), each with about 9.7% of the rows, which is about 680 customers.

**Insights:** The dataset has no rare categories. Even the smallest group has hundreds of customers, which is enough to estimate a churn rate. The two groups just below 10% are the same customers, because no phone service appears in both PhoneService and MultipleLines.

**Effect on the ML pipeline:** No grouping of rare categories is needed before encoding, and there is little risk that a category is missing in the training or test split. This makes the encoding step simple.

## 4. High Cardinality and Low Cardinality

**Concept:** Cardinality is the number of unique categories of a feature. Low cardinality means a few categories (for example 2 to 5) and these are easy to one-hot encode. High cardinality means many categories (for example more than 20 or 50, or most values are unique), and one-hot encoding would create too many columns. High cardinality features need other methods, such as target encoding, frequency encoding, hashing or grouping. An identifier column is the extreme case, where every value is unique.

**Example (AI/ML):** A zip code feature with 30000 unique values cannot be one-hot encoded for a small dataset. A solution is to replace each zip code with the average target of that zip code, which is called target encoding, with care to avoid leakage.

In [ ]:
card = df[cat_cols + ["customerID"]].nunique().sort_values()

def cardinality_level(n, total):
    if n / total > 0.5:
        return "identifier (every value unique)"
    if n > 20:
        return "high"
    if n > 5:
        return "medium"
    return "low"

card_table = pd.DataFrame({
    "unique_values": card,
    "unique_ratio": (card / len(df)).round(4),
    "level": [cardinality_level(n, len(df)) for n in card]
})
print(card_table)
print()
print("Number of columns one-hot encoding would create (all 16 features):", card.drop("customerID").sum())
print("With one category dropped per feature:", card.drop("customerID").sum() - len(cat_cols))
print("If customerID were one-hot encoded it alone would create:", card["customerID"], "columns")

**What the code does:** It counts the unique values of all categorical features and of customerID, classifies each as low, medium, high or identifier using simple thresholds, and calculates how many columns one-hot encoding would create.

**Why this method was used:** The number of columns created by encoding is the practical effect of cardinality, so calculating it shows directly whether one-hot encoding is a sensible choice.

**What the output means:** All sixteen categorical features have low cardinality (2 to 4 categories). One-hot encoding of all of them would create 43 columns, or 32 if one category is dropped per feature. The customerID column has 7043 unique values, one per row, so it is an identifier and not a real feature.

**Insights:** There is no high cardinality feature in this dataset, apart from the identifier. If customerID were encoded by mistake, it would create 7043 columns, which would be useless and would make the model memorize each customer.

**Effect on the ML pipeline:** One-hot encoding is safe and cheap for every feature here, and no special encoding for high cardinality is needed. The customerID column must be dropped before training.

## 5. Category Imbalance

**Concept:** Category imbalance means the categories of a feature (or of the target) have very different sizes. For an input feature it means that one label dominates. For the target it is a bigger problem, because the model sees many examples of the majority class and few of the minority class, and it may learn to ignore the minority. The imbalance ratio is the size of the largest category divided by the size of the smallest one.

**Example (business):** In fraud detection only about 0.1% of the transactions are fraud. A model that says not fraud for every transaction is 99.9% accurate and completely useless, so imbalance changes how a model must be built and measured.

In [ ]:
target_counts = df["Churn"].value_counts()
print("Target Churn:")
print(target_counts)
print("Percent:", (target_counts / len(df) * 100).round(1).to_dict())
print("Imbalance ratio:", round(target_counts.iloc[0] / target_counts.iloc[1], 2))
print("Accuracy of always predicting No:", round(target_counts.iloc[0] / len(df) * 100, 1), "%")
print()

print("Features where one category has more than 70 percent of the rows:")
dominant = dist_table[dist_table["largest_%"] > 70][["feature", "largest_category", "largest_%", "largest_to_smallest"]]
print(dominant.sort_values("largest_%", ascending=False).to_string(index=False))

**What the code does:** It prints the counts, percentages and imbalance ratio of the target Churn, and the accuracy that a model would get by always predicting No. Then it lists the input features where one category has more than 70% of the rows.

**Why this method was used:** The always predict No accuracy is the baseline that any model must beat, and it shows why accuracy alone is a weak metric here. The list of dominated features finds the input imbalance.

**What the output means:** The target has about 5174 customers who stayed and 1869 who left, which is a ratio of about 2.8 to 1. Always predicting No would give about 73.5% accuracy. Among the input features, PhoneService, SeniorCitizen and Dependents have a dominant category above 70%.

**Insights:** The imbalance of the target is moderate and not extreme, so it can be handled with simple methods. The input imbalance is mild, and it is mainly a concern for PhoneService.

**Effect on the ML pipeline:** The target imbalance means I should use metrics such as precision, recall, F1 or ROC AUC and not accuracy alone, use stratified splitting so that both the training set and the test set keep the same churn share, and maybe use class weights. Input imbalance needs no special treatment.

## 6. Relationship Between Categorical Features and the Target

**Concept:** To find which categorical features are related to the target, I compare the churn rate of the categories (done in Notebook 5), and I also measure the strength of the association with a single number. The chi-square statistic compares the real counts with the counts expected if the feature and the target were independent. Cramers V converts it to a value between 0 (no association) and 1 (perfect association) that does not depend on the number of rows or categories. As a rough guide, below 0.1 is very weak, 0.1 to 0.3 is weak to moderate and above 0.3 is strong.

**Example (business):** In a bank data set, the feature account type shows a default rate of 3% for one type and 15% for another, which gives a high Cramers V, while the feature customer ID parity (odd or even) shows 8% for both, which gives a value near 0.

In [ ]:
def cramers_v(x, y):
    table = pd.crosstab(x, y)
    n = table.values.sum()
    expected = np.outer(table.sum(axis=1), table.sum(axis=0)) / n
    chi2 = ((table.values - expected) ** 2 / expected).sum()
    k = min(table.shape) - 1
    return np.sqrt(chi2 / (n * k)), chi2

rows = []
for col in cat_cols:
    v, chi2 = cramers_v(df_cat[col], df_cat["Churn"])
    rows.append([col, round(chi2, 1), round(v, 3)])

assoc = pd.DataFrame(rows, columns=["feature", "chi_square", "cramers_v"]).sort_values("cramers_v", ascending=False)
print(assoc.to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 6))
plot_assoc = assoc.sort_values("cramers_v")
colors = ["indianred" if v >= 0.3 else "steelblue" if v >= 0.1 else "lightgray" for v in plot_assoc["cramers_v"]]
plt.barh(plot_assoc["feature"], plot_assoc["cramers_v"], color=colors)
plt.axvline(0.1, color="black", linestyle=":")
plt.axvline(0.3, color="black", linestyle="--")
plt.xlabel("Cramers V with Churn")
plt.title("Association of each categorical feature with Churn")
plt.tight_layout()
plt.show()

In [ ]:
# churn rate and customer count in the categories of the four strongest features
top4 = assoc["feature"].head(4).tolist()

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
overall = df_cat["Churn_num"].mean() * 100

for ax, col in zip(axes, top4):
    rate = df_cat.groupby(col)["Churn_num"].mean() * 100
    count = df_cat[col].value_counts()[rate.index]
    ax.bar(rate.index, rate.values, color="indianred")
    for i, (r, c) in enumerate(zip(rate.values, count.values)):
        ax.text(i, r + 1, "n=" + str(c), ha="center", fontsize=9)
    ax.axhline(overall, color="black", linestyle="--")
    ax.set_title(col)
    ax.set_ylabel("Churn rate (%)")
    ax.tick_params(axis="x", rotation=25)
    ax.set_ylim(0, 60)

plt.tight_layout()
plt.show()

**What the code does:** The first cell defines a function that builds the cross tabulation, calculates the chi-square statistic with numpy and converts it to Cramers V, and applies it to every categorical feature against Churn. The second cell draws the values as a bar chart with lines at 0.1 and 0.3. The third cell draws the churn rate by category of the four strongest features and writes the customer count on each bar.

**Why this method was used:** Churn rate tables are easy to read but cannot be compared across features with different numbers of categories. Cramers V gives one comparable number for each feature, and the counts on the bars show that the rates are based on enough customers.

**What the output means:** Contract has the highest Cramers V, above 0.4, and online security, tech support, internet service and payment method follow, between about 0.3 and 0.35. The add-on services such as online backup and device protection are weaker, and gender and PhoneService are close to 0. In the bar charts, month-to-month, no online security, no tech support, fiber optic and electronic check have much higher churn rates than the overall rate.

**Insights:** Contract, online security, tech support, internet service and payment method are the strongest categorical features, and gender and PhoneService are the weakest. The high rates are based on groups of thousands of customers, so they are reliable. Cramers V shows the strength of the association only, and not the direction or the cause.

**Effect on the ML pipeline:** The strongest features should be kept and encoded carefully. The weak features can be dropped or tested in feature selection. Association is not causation, so a feature such as electronic check may be a sign of another factor and not the cause of leaving.

## 7. Do the No Internet Service and No Phone Service Labels Behave Like No?

**Concept:** In Notebook 3 I noted that seven features have a third label (No internet service or No phone service) that looks like No. Before merging them with No, I should check if the churn behavior is the same. If it is, merging is safe. If it is different, merging would hide a real difference.

**Example (AI/ML):** In a survey, the answers No and Not applicable look similar, but if people who answered Not applicable behave very differently from those who answered No, then merging them would mix two different groups.

In [ ]:
addons = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

rows = []
for col in addons:
    rate = df_cat.groupby(col)["Churn_num"].mean() * 100
    count = df_cat[col].value_counts()
    for category in rate.index:
        rows.append([col, category, count[category], round(rate[category], 1)])

addon_table = pd.DataFrame(rows, columns=["feature", "category", "customers", "churn_rate"])
addon_table.pivot(index="feature", columns="category", values="churn_rate")

**What the code does:** For the seven features with a third label it calculates the churn rate of each category and shows them in a table with the features as rows and the categories as columns.

**Why this method was used:** Comparing churn rates is the direct test of whether the third label can be merged with No without losing information.

**What the output means:** In the six internet add-on features the churn rate of No internet service is very low, about 7%, while the rate of No is high, about 33% to 42% depending on the feature, and the rate of Yes is lower than No. In MultipleLines the rate of No phone service is close to the rate of No.

**Insights:** For the internet add-ons, No internet service customers behave very differently from the customers who have internet but did not take the add-on, so merging them into No would mix a low risk group with a high risk group. The reason is that this group is the same as InternetService = No, which is already a very low churn group. For MultipleLines the two labels behave almost the same, so merging them is safe.

**Effect on the ML pipeline:** In the add-on features, merging No internet service into No can weaken a linear model, unless InternetService is also used and the interaction is captured. Keeping three categories, or using a tree model, is safer. This changes my earlier thought in the data quality notebook that merging is a simple choice, and it needs to be tested during preprocessing.

## 8. Relationships Among Categorical Features

**Concept:** Categorical features can also be related to each other, in the same way as numerical features are correlated. Cramers V between two features measures how strongly one tells about the other. If two features have a very high association, they carry overlapping information, which is the categorical form of multicollinearity.

**Example (business):** In a retail dataset, the feature store type and the feature store size are probably strongly associated, because small shops are almost always of the same type. Using both adds little new information.

In [ ]:
v_matrix = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
for a in cat_cols:
    for b in cat_cols:
        if a == b:
            v_matrix.loc[a, b] = 1.0
        else:
            v_matrix.loc[a, b] = cramers_v(df_cat[a], df_cat[b])[0]

plt.figure(figsize=(13, 10))
sns.heatmap(v_matrix, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, annot_kws={"size": 7})
plt.title("Cramers V between categorical features")
plt.tight_layout()
plt.show()

In [ ]:
# strongest pairs
pairs = v_matrix.where(np.triu(np.ones(v_matrix.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print(pairs.head(12).round(2).to_string())

**What the code does:** It calculates Cramers V for every pair of categorical features, draws it as a heatmap, and lists the twelve strongest pairs.

**Why this method was used:** The same measure used for the target can be used between features, so I can find overlapping features before encoding.

**What the output means:** The strongest pairs are between InternetService and the six internet add-on features, and among the add-on features themselves. Payment method, contract and paperless billing show moderate associations. Gender is close to 0 with all other features. Partner and Dependents are moderately associated.

**Insights:** The internet add-on features are linked to InternetService because one of their three labels is determined by it (No internet service means InternetService is No). The add-on features also overlap with each other, since customers who buy one add-on often buy others. This is the categorical form of the multicollinearity found in the multivariate notebook.

**Effect on the ML pipeline:** Overlapping features make linear model coefficients unstable, so a smaller set of add-on features, or a combined feature such as the number of add-on services, could be tested in feature engineering. Tree models handle the overlap better.

# Findings

| Question | Finding |
|---|---|
| Unique categories | 16 categorical features with 2 to 4 categories each, and clean labels |
| Distribution | Gender is even, and PhoneService (about 90%) and SeniorCitizen (about 84%) are dominated by one label |
| Rare categories | None below 5%, and the smallest group (no phone service) is about 9.7% |
| Cardinality | All low. One-hot encoding creates 43 columns (32 with one dropped per feature). customerID is an identifier |
| Category imbalance | Target is about 73.5% No and 26.5% Yes, a ratio of about 2.8 to 1 |
| Strongest features for Churn | Contract, online security, tech support, internet service, payment method |
| Weakest features for Churn | gender, PhoneService |
| Third labels | No internet service is very different from No in the six add-on features, but No phone service is close to No in MultipleLines |
| Overlapping features | InternetService and the six add-ons, and the add-ons with each other |

# Summary

- The categorical features are clean, low in cardinality and have no rare categories.
- The target is moderately imbalanced, and a stratified split and suitable metrics are needed.
- Contract is the strongest categorical feature for churn, followed by online security, tech support, internet service and payment method.
- Gender and PhoneService show almost no relation with churn.
- The No internet service label should not be merged blindly with No.
- The add-on features overlap with InternetService and with each other.

**Points for later sprints**
1. Use one-hot encoding for nominal features such as PaymentMethod, and consider ordinal encoding for Contract.
2. Drop customerID, and test dropping the weak features gender and PhoneService.
3. Test keeping and merging the No internet service label for the add-on features.
4. Use a stratified train and test split because of the target imbalance.
5. Consider a combined feature such as the number of add-on services.

**What I learned:** Cardinality decides the encoding method, and a third label that looks like No can hide a real group difference. Cramers V puts the strength of the relationship with the target into one comparable number, and it works for features with a different number of categories.

Next: `10_Target_Analysis.ipynb`